# Summary length on Key-to-Door

The summary-length ablation of the study protocol: the method RSM (`raw_summary`, the replacing write) with M ∈ {1, 4, 8, 16} summary tokens at C = 32, on the Dark Key-to-Door confirmation roster (256 tasks × seeds 42 / 100 / 2026), frozen 8M endpoints, one 4,000-step task per roster task. M = 4 is the paper's cell; every other arm is its own study root.

**(a)** Doors per 500-step window over the 4,000-step task, one curve per M; the band is the joint seed/task 95 % interval of the paper's M = 4 (the other arms' uncertainty is read as paired contrasts in (b)); the dotted line marks the trained budget (500 steps). **(b)** The declared paired contrast RSM M − RSM M4 (same task and seed) in the trained window (steps 1–500) and in the last window (steps 3,501–4,000): joint interval and the three seeds; the grey band is ±δ (δ = 1 door). Tick labels carry each arm's persistent state per actor.

Nothing is recomputed: the curves and contrasts are the saved read of `scripts/capacity_read.py` under `<8M root>/reports/dark-key-to-door/confirmation/capacity/raw_summary/`, and the state is each arm's `systems.json`. The figure is drawn at the ICLR text width (5.5 in) with 7-pt type, so it is included at `width=\linewidth`.

In [ ]:
"""Locate the repository, the study roots and the output folders.

Environment overrides, for a copy of the records elsewhere:
  REASONED_ICRL_OUTPUTS   study roots (default <repo>/outputs)
  FIGURE_OUTPUT_ROOT      where the figure is written (default notebooks/outputs/figures);
                          its tables and notes go to the sibling data/ folder
"""
import os
import sys
from pathlib import Path

from IPython.display import Markdown, display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO / "notebooks"))

from figures_common import default_layout, markdown_preview  # noqa: E402

OUTPUTS = Path(os.environ.get("REASONED_ICRL_OUTPUTS", REPO / "outputs"))
FIGURES_DIR = Path(
    os.environ.get("FIGURE_OUTPUT_ROOT", REPO / "notebooks/outputs/figures")
)
DATA_DIR = FIGURES_DIR.parent / "data"
layout = default_layout(REPO, outputs=OUTPUTS)
print(f"records under {OUTPUTS}\nfigures to  {FIGURES_DIR}\ntables to   {DATA_DIR}")

In [ ]:
import csv
import json
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
from figures_common import (
    GRID,
    INK,
    INK_FAINT,
    INK_MUTED,
    METHOD_COLOUR,
    METHOD_CONDITION,
    BuiltFigure,
    StudyLayout,
    allocated_state_bytes,
    style_axes,
)
from matplotlib.axes import Axes
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

TEXT_WIDTH = 5.5
"""The ICLR text width in inches: the figure prints at the size it is drawn."""
PAPER_TOKENS = 4
"""The paper's summary length; every other arm is read against it."""
ARM_STUDIES: dict[int, str] = {
    1: "keydoor-capacity-m1-8m",
    4: "summary-memory-8m",
    8: "keydoor-capacity-m8-8m",
    16: "keydoor-capacity-8m",
}
"""Study root per summary length."""
SHADES: dict[int, str] = {1: "#F4A582", 4: METHOD_COLOUR, 8: "#A63603", 16: "#5E1F02"}
"""One vermillion ramp ordered by M; the paper's cell keeps the method's colour."""
MARKERS: dict[int, str] = {1: "v", 4: "o", 8: "s", 16: "D"}
"""A marker per M, so the arms stay apart in grayscale."""
SEEDS: tuple[int, ...] = (42, 100, 2026)
HORIZON = 4000
WINDOW = 500
TRAINED = 500
DELTA = 1.0
MEASURES: tuple[tuple[str, str], ...] = (
    ("first window", "Steps 1–500 (trained)"),
    ("last window", "Steps 3,501–4,000"),
)

Row = dict[str, object]


@dataclass(frozen=True)
class SummaryLengthOptions:
    """``cell`` names the summary carrier read (the method); ``split`` the roster
    of the saved read; ``y_limits`` the doors axis of panel (a)."""

    cell: str = METHOD_CONDITION
    split: str = "confirmation"
    y_limits: tuple[float, float] = (20.0, 50.0)


def print_rc() -> dict[str, object]:
    """Matplotlib settings for a figure drawn at :data:`TEXT_WIDTH` (7-pt type),
    the print style of the paper's other figures."""
    return {
        "font.family": "sans-serif",
        "font.sans-serif": ["Liberation Sans", "Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 7.0,
        "axes.labelsize": 7.0,
        "axes.titlesize": 7.0,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 3.0,
        "axes.labelpad": 2.0,
        "legend.fontsize": 6.5,
        "legend.frameon": False,
        "legend.handletextpad": 0.5,
        "legend.borderaxespad": 0.3,
        "legend.labelspacing": 0.3,
        "xtick.labelsize": 6.3,
        "ytick.labelsize": 6.3,
        "xtick.major.size": 2.0,
        "ytick.major.size": 2.0,
        "xtick.major.width": 0.5,
        "ytick.major.width": 0.5,
        "xtick.major.pad": 1.5,
        "ytick.major.pad": 1.5,
        "xtick.color": INK_MUTED,
        "ytick.color": INK_MUTED,
        "xtick.labelcolor": INK,
        "ytick.labelcolor": INK,
        "axes.edgecolor": INK_MUTED,
        "axes.linewidth": 0.5,
        "axes.labelcolor": INK,
        "text.color": INK,
        "lines.linewidth": 1.0,
        "lines.markersize": 2.8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.dpi": 150,
    }


def read_rows(path: Path) -> list[Row]:
    if not path.is_file():
        return []
    with path.open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))


def number(value: object) -> float:
    return float(str(value))


def capacity_directory(layout: StudyLayout, options: SummaryLengthOptions) -> Path:
    """``<8M root>/reports/dark-key-to-door/<split>/capacity/<cell>``, where
    ``scripts/capacity_read.py`` writes the paired read."""
    root = layout.roots[("keydoor", METHOD_CONDITION)]
    return (
        root
        / "reports"
        / "dark-key-to-door"
        / options.split
        / "capacity"
        / options.cell
    )


def state_bytes(
    layout: StudyLayout, options: SummaryLengthOptions, sources: list[str]
) -> dict[int, int]:
    """Each arm's persistent state per actor (seed 42's ``systems.json``; the
    bytes depend on M alone)."""
    base = layout.roots[("keydoor", METHOD_CONDITION)].parent
    protocol = layout.spec("keydoor").protocol
    found: dict[int, int] = {}
    for tokens, study in ARM_STUDIES.items():
        path = base / study / protocol / options.cell / "seed-42" / "systems.json"
        if path.is_file():
            found[tokens] = allocated_state_bytes(json.loads(path.read_text()))[0]
            sources.append(str(path))
    return found


def window_curves(rows: list[Row], cell: str) -> dict[int, list[Row]]:
    """The 4,000-step windows per M, in window order."""
    curves: dict[int, list[Row]] = {}
    for row in rows:
        if row.get("cell") != cell or int(number(row["horizon"])) != HORIZON:
            continue
        curves.setdefault(int(number(row["memory_tokens"])), []).append(row)
    return {
        m: sorted(r, key=lambda x: int(number(x["window"]))) for m, r in curves.items()
    }


def paired_contrasts(rows: list[Row]) -> dict[tuple[int, str], Row]:
    """RSM M − RSM M4 at the longest horizon, first and last window."""
    return {
        (int(number(r["memory_tokens"])), str(r["measure"])): r
        for r in rows
        if int(number(r["horizon"])) == HORIZON and str(r["measure"]) in dict(MEASURES)
    }


def draw_windows(
    ax: Axes, curves: dict[int, list[Row]], options: SummaryLengthOptions
) -> None:
    style_axes(ax)
    ax.axvline(TRAINED, color=INK_FAINT, linewidth=0.6, linestyle=(0, (1, 1.5)))
    ax.text(
        TRAINED - 50,
        options.y_limits[0] + 0.8,
        "trained\nbudget",
        color=INK_MUTED,
        fontsize=6.0,
        va="bottom",
        ha="right",
        linespacing=1.0,
    )
    for tokens in sorted(curves):
        rows = curves[tokens]
        x = [(number(r["window"]) - 0.5) * WINDOW for r in rows]
        paper = tokens == PAPER_TOKENS
        if paper:
            # One band: the arms overlap within their intervals, and four
            # stacked bands hide the curves; (b) reads each arm's uncertainty.
            ax.fill_between(
                x,
                [number(r["lower"]) for r in rows],
                [number(r["upper"]) for r in rows],
                color=SHADES[tokens],
                alpha=0.14,
                linewidth=0,
                zorder=1,
            )
        ax.plot(
            x,
            [number(r["estimate"]) for r in rows],
            color=SHADES[tokens],
            marker=MARKERS[tokens],
            linewidth=1.3 if paper else 0.9,
            markersize=3.0 if paper else 2.6,
            zorder=4 if paper else 3,
        )
    ax.set_xlim(0, HORIZON)
    ax.set_xticks([0, 1000, 2000, 3000, 4000])
    ax.set_xticklabels(["0", "1,000", "2,000", "3,000", "4,000"])
    ax.set_ylim(*options.y_limits)
    ax.set_xlabel("Steps in the task")
    ax.set_ylabel("Doors per 500 steps")
    ax.set_title("(a) Adaptation over one long task")


def draw_contrasts(
    ax: Axes, contrasts: dict[tuple[int, str], Row], states: dict[int, int]
) -> list[int]:
    style_axes(ax)
    arms = sorted(m for m in ARM_STUDIES if m != PAPER_TOKENS)
    ax.axhspan(-DELTA, DELTA, color=GRID, linewidth=0, zorder=0)
    ax.axhline(0.0, color=INK_MUTED, linewidth=0.6, zorder=1)
    offsets = {"first window": -0.14, "last window": 0.14}
    for position, tokens in enumerate(arms):
        for measure, _ in MEASURES:
            row = contrasts.get((tokens, measure))
            if row is None:
                continue
            x = position + offsets[measure]
            estimate = number(row["estimate"])
            low, high = number(row["lower"]), number(row["upper"])
            seeds = [
                number(row[f"seed_{s}"])
                for s in SEEDS
                if row.get(f"seed_{s}") not in (None, "")
            ]
            ax.scatter(
                [x + d for d in (-0.05, 0.0, 0.05)][: len(seeds)],
                seeds,
                s=4,
                color=SHADES[tokens],
                alpha=0.45,
                linewidths=0,
                zorder=2,
            )
            ax.errorbar(
                [x],
                [estimate],
                yerr=[[estimate - low], [high - estimate]],
                fmt=MARKERS[tokens],
                color=SHADES[tokens],
                ecolor=SHADES[tokens],
                elinewidth=0.8,
                capsize=1.6,
                capthick=0.6,
                markersize=3.6,
                markerfacecolor=SHADES[tokens] if measure == "last window" else "white",
                markeredgewidth=0.8,
                zorder=3,
            )
    ax.set_xticks(range(len(arms)))
    ax.set_xticklabels(
        [
            f"M = {m}" + (f"\n{states[m] / 1000:.0f} KB" if m in states else "")
            for m in arms
        ]
    )
    ax.set_xlim(-0.6, len(arms) - 0.4)
    ax.set_ylabel("Difference in doors per 500 steps")
    ax.set_title(f"(b) Paired difference to M = {PAPER_TOKENS}")
    return arms


def build_summary_length_figure(
    layout: StudyLayout, options: SummaryLengthOptions = SummaryLengthOptions()
) -> BuiltFigure:
    """The two-panel figure, its tables and notes, from the saved read."""
    directory = capacity_directory(layout, options)
    sources: list[str] = []
    windows = read_rows(directory / "windows.csv")
    contrast_rows = read_rows(directory / "contrasts.csv")
    for name, rows in (("windows.csv", windows), ("contrasts.csv", contrast_rows)):
        if rows:
            sources.append(str(directory / name))
    curves = window_curves(windows, options.cell)
    contrasts = paired_contrasts(contrast_rows)
    states = state_bytes(layout, options, sources)
    pending = [m for m in ARM_STUDIES if m not in curves]
    with plt.rc_context(print_rc()):
        figure, (left, right) = plt.subplots(
            1,
            2,
            figsize=(TEXT_WIDTH, 2.05),
            width_ratios=(1.25, 1.0),
            layout="constrained",
        )
        draw_windows(left, curves, options)
        arms = draw_contrasts(right, contrasts, states)
        handles = [
            Line2D(
                [],
                [],
                color=SHADES[m],
                marker=MARKERS[m],
                linewidth=1.3 if m == PAPER_TOKENS else 0.9,
                label=f"M = {m}" + (" (paper)" if m == PAPER_TOKENS else ""),
            )
            for m in ARM_STUDIES
            if m in curves
        ]
        # Nothing missing is written on a chart: a pending arm is named in the
        # legend title only.
        left.legend(
            handles=handles,
            loc="lower right",
            ncol=2,
            columnspacing=1.0,
            handlelength=1.8,
            title=("Records pending: M = " + ", ".join(map(str, pending)))
            if pending
            else None,
            title_fontsize=6.0,
        )
        right.legend(
            handles=[
                Line2D(
                    [],
                    [],
                    color=INK_MUTED,
                    marker="o",
                    linestyle="none",
                    markerfacecolor="white",
                    markersize=3.6,
                    label=MEASURES[0][1],
                ),
                Line2D(
                    [],
                    [],
                    color=INK_MUTED,
                    marker="o",
                    linestyle="none",
                    markersize=3.6,
                    label=MEASURES[1][1],
                ),
                Patch(facecolor=GRID, edgecolor="none", label=f"±δ ({DELTA:g} door)"),
            ],
            loc="lower right",
            handlelength=1.2,
        )
        low, high = right.get_ylim()
        right.set_ylim(min(-6.0, low), max(7.0, high))
    window_table: list[Row] = [
        {
            "memory_tokens": m,
            "window": int(number(r["window"])),
            "estimate": number(r["estimate"]),
            "lower": number(r["lower"]),
            "upper": number(r["upper"]),
            **{f"seed_{s}": r.get(f"seed_{s}") for s in SEEDS},
        }
        for m, rows in sorted(curves.items())
        for r in rows
    ]
    contrast_table: list[Row] = [
        {
            "memory_tokens": m,
            "state_bytes": states.get(m),
            "measure": measure,
            "estimate": number(row["estimate"]),
            "lower": number(row["lower"]),
            "upper": number(row["upper"]),
            **{f"seed_{s}": row.get(f"seed_{s}") for s in SEEDS},
            "within_delta_every_seed": row.get("within_delta_every_seed"),
            "gain_every_seed": row.get("gain_every_seed"),
            "loss_every_seed": row.get("loss_every_seed"),
        }
        for (m, measure), row in sorted(contrasts.items())
    ]
    notes: dict[str, object] = {
        "cell": options.cell,
        "split": options.split,
        "horizon_steps": HORIZON,
        "window_steps": WINDOW,
        "delta_doors": DELTA,
        "arms_drawn": sorted(curves),
        "arms_pending": pending,
        "contrast_arms": arms,
        "state_bytes": states,
        "intervals": "joint seed/task bootstrap, 95 %, scripts/capacity_read.py",
        "capacity_read": str(directory),
    }
    return BuiltFigure(
        name="figure_summary_length",
        figure=figure,
        tables={"windows": window_table, "contrasts": contrast_table},
        notes=notes,
        sources=sources,
    )

In [ ]:
# Parameters: `cell` the summary carrier read, `split` the roster of the saved read,
# `y_limits` the doors axis of panel (a).
options = SummaryLengthOptions()
summary_length = build_summary_length_figure(layout, options)
display(summary_length.figure)

In [ ]:
# What was read, and which arms are pending.
print(json.dumps(summary_length.notes, indent=2, default=str))

In [ ]:
# The paired contrasts as drawn in panel (b).
display(Markdown(markdown_preview(summary_length.tables["contrasts"], limit=12)))

In [ ]:
# Save: the figure (PDF + PNG), the plotted tables and the notes.
for path in summary_length.save(FIGURES_DIR, DATA_DIR):
    print(path.relative_to(REPO) if path.is_relative_to(REPO) else path)
summary_length.close()